# 08. Vector DB & Embedding: Qdrant 영구 저장과 GUI 확인

## 학습 목표

- `공직자_민원응대_핵심_매뉴얼.pdf`를 **medium chunk** 전략으로 분할한다.
- `OpenAIEmbeddings`로 문서를 임베딩한다.
- 임베딩 데이터를 **Qdrant Server**에 영구 저장한다.
- Qdrant Web UI에서 collection, point, payload, metadata를 확인한다.
- 이미 저장된 Qdrant collection을 재사용하여 검색한다.

## 최종 구조

```text
PDF 로드
↓
텍스트 정제
↓
medium chunk 분할
chunk_size=700, chunk_overlap=120
↓
OpenAIEmbeddings 임베딩
↓
Docker 기반 Qdrant Server 저장
↓
Qdrant Web UI에서 metadata 확인
↓
기존 collection 재사용 검색
```

## 0. 사전 작업 1: Python 패키지 설치

### uv 사용 시

```powershell
uv add langchain-qdrant qdrant-client 
```

## 0. 사전 작업 2: Qdrant Server 실행
```text
# 도커 설치
https://www.docker.com/
```

임베딩 데이터를 계속 보관하고 GUI에서 확인하기 위해 **Docker 기반 Qdrant Server**를 사용한다.

아래 명령은 **Windows PowerShell**에서 실행한다.

```powershell
# 1) Qdrant 이미지 다운로드
docker pull qdrant/qdrant

# 2) 영구 저장용 Docker volume 생성
docker volume create qdrant_storage

# 3) 기존 컨테이너가 있다면 정리
docker rm -f qdrant-rag

# 4) Qdrant Server 실행
docker run -d `
  --name qdrant-rag `
  -p 6333:6333 `
  -p 6334:6334 `
  -v qdrant_storage:/qdrant/storage `
  qdrant/qdrant
```

실행 확인:

```powershell
docker ps
```

Web UI 접속:

```text
http://localhost:6333/dashboard
```

> `qdrant_storage` Docker volume을 사용하므로 컨테이너를 재시작해도 collection 데이터가 유지된다.

## 1. 라이브러리 import 및 기본 설정

`PDF_PATH`는 현재 노트북이 `notebooks` 또는 실습 폴더 안에 있고, 데이터가 `../data`에 있다고 가정한다.

In [1]:
from pathlib import Path
import os, re
from pprint import pprint

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
from qdrant_client import models

In [9]:
# .env 탐색: 프로젝트 구조에 따라 위치가 다를 수 있으므로 후보 경로를 순차 확인한다.
load_dotenv(dotenv_path="../../.env", override=True)
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

PDF_PATH = Path("../data/공직자_민원응대_핵심_매뉴얼.pdf")

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "civil_complaint_manual_medium"

EMBEDDING_MODEL = "text-embedding-3-small"
CHUNK_SIZE = 700
CHUNK_OVERLAP = 120

# True: collection을 삭제 후 새로 생성한다.
# False: 이미 저장된 collection을 재사용한다.
RECREATE_COLLECTION = True

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

OPENAI_API_KEY exists: True


## 2. Qdrant Server 연결 확인

이 셀이 실패하면 Python 코드 문제가 아니라 Qdrant 컨테이너가 실행되지 않았을 가능성이 높다.

In [3]:
client = QdrantClient(url=QDRANT_URL, timeout=10)

try:
    collections = client.get_collections().collections
    print("Qdrant 연결 성공")
    print("현재 collection 목록:")
    for collection in collections:
        print("-", collection.name)
except Exception as e:
    raise RuntimeError(
        "Qdrant Server에 연결할 수 없습니다. "
        "Docker Desktop이 실행 중인지, qdrant-rag 컨테이너가 실행 중인지 확인하세요. "
        "Web UI: http://localhost:6333/dashboard"
    ) 

Qdrant 연결 성공
현재 collection 목록:
- civil_service_manual


## 3. PDF 로드 및 텍스트 정제 함수

PDF 페이지를 LangChain의 `Document` 객체로 변환한다.  
각 페이지에는 `source`, `page`, `document_type` metadata를 부여한다.

In [4]:
def clean_text(text: str) -> str:
    """PDF에서 추출된 텍스트를 간단히 정제한다."""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_pdf_pages(pdf_path: Path) -> list[Document]:
    """PDF를 페이지 단위 Document 리스트로 로드한다."""
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    page_docs = []

    for doc in docs:
        text = clean_text(doc.page_content or "")
        if not text:
            continue

        page_docs.append(
            Document(
                page_content=text,
                metadata={
                    **doc.metadata,
                    "source": pdf_path.name,
                    "page": doc.metadata.get("page", 0) + 1, 
                }
            )
        )
    return page_docs

## 4. medium chunk 분할 함수

이전 실험 결과를 기준으로 최종 전략은 다음과 같이 고정한다.

| 항목 | 값 |
|---|---:|
| chunk_size | 700 |
| chunk_overlap | 120 |
| chunk_strategy | medium |

이 설정은 작은 청크보다 문맥 보존력이 높고, 큰 청크보다 불필요한 정보 혼입이 적다.

In [5]:
def make_medium_chunks(
    docs: list[Document],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[Document]:
    """매뉴얼형 문서에 적합한 medium chunk를 생성한다."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "STEP", "‣", "•", ". ", " "],
    )

    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata.update(
            {
                "chunk_id": i,
                "chunk_size": chunk_size,
                "chunk_overlap": chunk_overlap,
                "chunk_strategy": "medium",
                "embedding_model": EMBEDDING_MODEL,
                "collection_name": COLLECTION_NAME,
            }
        )

    return chunks

## 5. chunk 개수 및 샘플 metadata 확인

청크 개수와 샘플 metadata를 확인한다.

In [8]:
pages = load_pdf_pages(PDF_PATH)
medium_chunks = make_medium_chunks(pages)

print("페이지 수:", len(pages))
print("medium chunk 수:", len(medium_chunks))

print("\n[첫 번째 chunk metadata]")
pprint(medium_chunks[0].metadata)


페이지 수: 20
medium chunk 수: 43

[첫 번째 chunk metadata]
{'chunk_id': 0,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'medium',
 'collection_name': 'civil_complaint_manual_medium',
 'creationdate': '2025-11-30T19:46:56+09:00',
 'creator': 'Adobe InDesign CC 2014 (Windows)',
 'embedding_model': 'text-embedding-3-small',
 'moddate': '2025-11-30T19:47:04+09:00',
 'page': 1,
 'page_label': '1',
 'producer': 'Adobe PDF Library 11.0',
 'source': '공직자_민원응대_핵심_매뉴얼.pdf',
 'total_pages': 20,
 'trapped': '/False'}


In [ ]:
# BASE_DIR = Path.cwd().resolve().parent
# DATA_DIR = BASE_DIR / "data"
# DATA_DIR

WindowsPath('E:/langgraph_modular_rag/6.modular_rag_manual/data')

## 6. Qdrant collection 생성 또는 재사용

실습에서는 두 가지 모드를 사용한다.

| 모드 | 설정 | 설명 |
|---|---|---|
| 재생성 모드 | `RECREATE_COLLECTION = True` | 기존 collection을 삭제하고 새로 임베딩한다 |
| 재사용 모드 | `RECREATE_COLLECTION = False` | 이미 저장된 collection에 연결만 한다 |

처음 실습할 때는 `True`로 실행한다.  
한 번 저장한 뒤에는 `False`로 바꾸면 임베딩 비용을 다시 쓰지 않고 collection을 재사용할 수 있다.

In [11]:
def collection_exists(client: QdrantClient, collection_name: str) -> bool:
    """Qdrant collection 존재 여부를 확인한다."""
    try:
        return client.collection_exists(collection_name)
    except AttributeError:
        names = [c.name for c in client.get_collections().collections]
        return collection_name in names


exists = collection_exists(client, COLLECTION_NAME)
print(f"collection exists before run: {exists}")

if RECREATE_COLLECTION:
    print("collection을 새로 생성한다. 기존 collection이 있으면 삭제 후 재생성한다.")

    vector_store = QdrantVectorStore.from_documents(
        documents=medium_chunks,
        embedding=embeddings,
        url=QDRANT_URL,
        collection_name=COLLECTION_NAME,
        force_recreate=True,
    )
else:
    if not exists:
        raise ValueError(
            f"'{COLLECTION_NAME}' collection이 아직 없습니다. "
            "처음 실행할 때는 RECREATE_COLLECTION = True로 설정하세요."
        )

    print("기존 collection을 재사용한다. 새 임베딩을 생성하지 않는다.")

    vector_store = QdrantVectorStore.from_existing_collection(
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        url=QDRANT_URL,
    )

print("vector_store 준비 완료")

collection exists before run: False
collection을 새로 생성한다. 기존 collection이 있으면 삭제 후 재생성한다.
vector_store 준비 완료


## 7. Qdrant collection 정보 확인

저장된 point 수와 collection 설정을 확인한다.

In [12]:
collection_info = client.get_collection(COLLECTION_NAME)

print("collection name:", COLLECTION_NAME)
print("points count:", collection_info.points_count)
print("vectors count:", collection_info.indexed_vectors_count)
print("status:", collection_info.status)
print("\n[collection config]")
pprint(collection_info.config)

collection name: civil_complaint_manual_medium
points count: 43
vectors count: 0
status: green

[collection config]
CollectionConfig(params=CollectionParams(vectors={'': VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors={}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_

## 8. Qdrant Web UI에서 metadata 확인하기

브라우저에서 아래 주소로 접속한다.

```text
http://localhost:6333/dashboard
```

확인 순서:

```text
Collections
→ civil_complaint_manual_medium 선택
→ Points 또는 Scroll 메뉴 확인
→ payload 확인
→ page_content와 metadata 확인
```

LangChain이 Qdrant에 저장하는 기본 payload 구조는 대체로 다음과 같다.

```json
{
  "page_content": "청크 원문 텍스트",
  "metadata": {
    "source": "공직자_민원응대_핵심_매뉴얼.pdf",
    "page": 1,
    "chunk_id": 0,
    "chunk_size": 700,
    "chunk_overlap": 120,
    "chunk_strategy": "medium",
    "embedding_model": "text-embedding-3-small"
  }
}
```

## 9. Python으로 Qdrant payload 직접 확인

GUI와 동일하게 Python에서도 Qdrant point의 payload를 확인할 수 있다.  
`with_vectors=False`로 설정하면 긴 벡터값은 출력하지 않고 metadata만 확인할 수 있다.

In [13]:
points, next_page_offset = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=3,
    with_payload=True,
    with_vectors=False,
)

for i, point in enumerate(points, start=1):
    print("=" * 80)
    print(f"[{i}] point id:", point.id)
    print("payload keys:", list(point.payload.keys()))
    print("\nmetadata:")
    pprint(point.payload.get("metadata"))
    print("\npage_content preview:")
    print(point.payload.get("page_content", "")[:500].replace("\n", " "))

print("\nnext_page_offset:", next_page_offset)

[1] point id: 0320826b-f36a-44fb-8de5-2c410ef1ad06
payload keys: ['page_content', 'metadata']

metadata:
{'chunk_id': 15,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'medium',
 'collection_name': 'civil_complaint_manual_medium',
 'creationdate': '2025-11-30T19:46:56+09:00',
 'creator': 'Adobe InDesign CC 2014 (Windows)',
 'embedding_model': 'text-embedding-3-small',
 'moddate': '2025-11-30T19:47:04+09:00',
 'page': 10,
 'page_label': '10',
 'producer': 'Adobe PDF Library 11.0',
 'source': '공직자_민원응대_핵심_매뉴얼.pdf',
 'total_pages': 20,
 'trapped': '/False'}

page_content preview:
면담 후 60분 이내 휴게시간 부여(필요시 추가 부여)  담당자(팀장)는 위법행위 관리대장(특이민원 발생보고서) 작성 ‣ 부서장이 전담부서와 협의하여 법적 조치 등 필요사항 조치   녹음파일 청취 및 상황파악, 특이민원 발생보고서 검토  ※ 필요시, 서면 경고문 발송(우편, SMS)  고소, 고발 등 법적 조치사항 등에 대해 법적대응 전담부서 협의 3단계 상황 보고 4단계 부서장 대응 폭행3-2 ‣ 부서장은 피해직원 면담 및 치료 등 최선의 조치   [폭행이 발생하지 않은 경우] 피해공무원과 면담 후 휴게시간 부여  (또는 조기귀가 조치)하고, 필요에 따라 공무상 병가 등 조치  [폭행이 발생한 경우] 우선 조기 귀가(공무상 병가) 조치하고, 피해 상황에   따라 필요한 추가 조치(심리상담, 보직변경, 공무

## 10. similarity search 테스트

이전 노트북과 동일한 질문으로 검색한다.

In [15]:
question = "전화 중 욕설을 하면 어떻게 대응해야 하나요?"

results = vector_store.similarity_search_with_score(question, k=3)

for i, (doc, score) in enumerate(results, start=1):
    print("=" * 80)
    print(f"[{i}] score: {score}")
    print("metadata:")
    pprint(doc.metadata)
    print("\ncontent preview:")
    print(doc.page_content[:700].replace("\n", " "))

[1] score: 0.53228664
metadata:
{'_collection_name': 'civil_complaint_manual_medium',
 '_id': 'bfa61ef6-8f1a-433b-9fe5-aa2df965a524',
 'chunk_id': 29,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'medium',
 'collection_name': 'civil_complaint_manual_medium',
 'creationdate': '2025-11-30T19:46:56+09:00',
 'creator': 'Adobe InDesign CC 2014 (Windows)',
 'embedding_model': 'text-embedding-3-small',
 'moddate': '2025-11-30T19:47:04+09:00',
 'page': 16,
 'page_label': '16',
 'producer': 'Adobe PDF Library 11.0',
 'source': '공직자_민원응대_핵심_매뉴얼.pdf',
 'total_pages': 20,
 'trapped': '/False'}

content preview:
질의 5 권장시간이 되면 전화를 무조건 끊어야 하는지? • 인·허가 등 정상적인 업무처리 과정에서 민원인과의 상담이 길어지는 경우는 정당한 사유없는 장시간   전화에 해당하지 않으므로 종결대상에 해당하지 않음. 질의 3 상급자 응대 후에도 계속 전화해서 폭언을 하는 경우는 어떻게 대응 해야 하는지? • 담당자에게 휴게시간 부여 등 응대를 중지하도록 조치하고 대행자 등은 다른 민원처리를 위해 해당   민원은 일시적으로 응대 보류함. • 폭언, 업무방해 등에 대해 경고문 발송, 고소·고발 등 법적 대응을 추진하여 위법행위를 지속하지   않도록 강력 대응.
[2] score: 0.5202817
metadata:
{'_collection_name': 'civil_compl

## 11. metadata filter 검색

Qdrant는 payload 기반 필터링이 가능하다.  
아래 예시는 `chunk_strategy == medium`인 point만 검색 대상으로 제한한다.

In [16]:
filtered_results = vector_store.similarity_search_with_score(
    query=question,
    k=3,
    filter=models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.chunk_strategy",
                match=models.MatchValue(value="medium"),
            )
        ]
    ),
)

for i, (doc, score) in enumerate(filtered_results, start=1):
    print("=" * 80)
    print(f"[{i}] score: {score}")
    print("page:", doc.metadata.get("page"))
    print("chunk_id:", doc.metadata.get("chunk_id"))
    print("chunk_strategy:", doc.metadata.get("chunk_strategy"))
    print(doc.page_content[:500].replace("\n", " "))

[1] score: 0.53228664
page: 16
chunk_id: 29
chunk_strategy: medium
질의 5 권장시간이 되면 전화를 무조건 끊어야 하는지? • 인·허가 등 정상적인 업무처리 과정에서 민원인과의 상담이 길어지는 경우는 정당한 사유없는 장시간   전화에 해당하지 않으므로 종결대상에 해당하지 않음. 질의 3 상급자 응대 후에도 계속 전화해서 폭언을 하는 경우는 어떻게 대응 해야 하는지? • 담당자에게 휴게시간 부여 등 응대를 중지하도록 조치하고 대행자 등은 다른 민원처리를 위해 해당   민원은 일시적으로 응대 보류함. • 폭언, 업무방해 등에 대해 경고문 발송, 고소·고발 등 법적 대응을 추진하여 위법행위를 지속하지   않도록 강력 대응.
[2] score: 0.5202817
page: 9
chunk_id: 12
chunk_strategy: medium
‣ 통화곤란 안내 및 통화 종료   선생님 지속적인 폭언 등으로 더 이상 상담이 어렵습니다.   통화를 종료하겠습니다.  [동일 민원인과 재통화 시] 민원인이 진정된 경우 정상 응대, 폭언을   지속할 경우 통화 종료 후 상급자가 응대·경고  담당자 상급자를 통해 전화주신 번호로 연락을 취할 수 있도록 하겠습니다.   통화 종료하겠습니다.   상급자 요청하신 사항에 대해서는 (~조치하겠습니다./~이유로 처리가   안되는 점 양해바랍니다.) 다만, 저희 직원에게 하신 폭언 등은 관련   법령에 따라 처벌받을 수 있으니 삼가주시기 바랍니다. 2단계 통화 종료 핵심대응 전화 종료 및 증거확보를 통한 후속조치(법적조치 등)로 경각심 부여 ‣ 민원전화 수신 시부터 녹음하여 위법행위 증거 확보   폭언·성희롱 등 위법행위에 대한 처벌 가능성 고지  선생님, 폭언 등을 계속하시면 정상적인 상담이 어렵고 관련 법령에 따라   처벌받을 수 있습니다. 1단계 증거 확보 ‣ 부서장 보고, 부서장은 피해
[3] score: 0.5028752
page: 8
chunk_id: 10
chunk_strateg

## 12. Retriever로 변환

RAG 체인에서 사용할 때는 vector store를 retriever로 변환한다.

In [17]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"[{i}] metadata")
    pprint(doc.metadata)
    print("\ncontent preview")
    print(doc.page_content[:500].replace("\n", " "))

[1] metadata
{'_collection_name': 'civil_complaint_manual_medium',
 '_id': 'bfa61ef6-8f1a-433b-9fe5-aa2df965a524',
 'chunk_id': 29,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'medium',
 'collection_name': 'civil_complaint_manual_medium',
 'creationdate': '2025-11-30T19:46:56+09:00',
 'creator': 'Adobe InDesign CC 2014 (Windows)',
 'embedding_model': 'text-embedding-3-small',
 'moddate': '2025-11-30T19:47:04+09:00',
 'page': 16,
 'page_label': '16',
 'producer': 'Adobe PDF Library 11.0',
 'source': '공직자_민원응대_핵심_매뉴얼.pdf',
 'total_pages': 20,
 'trapped': '/False'}

content preview
질의 5 권장시간이 되면 전화를 무조건 끊어야 하는지? • 인·허가 등 정상적인 업무처리 과정에서 민원인과의 상담이 길어지는 경우는 정당한 사유없는 장시간   전화에 해당하지 않으므로 종결대상에 해당하지 않음. 질의 3 상급자 응대 후에도 계속 전화해서 폭언을 하는 경우는 어떻게 대응 해야 하는지? • 담당자에게 휴게시간 부여 등 응대를 중지하도록 조치하고 대행자 등은 다른 민원처리를 위해 해당   민원은 일시적으로 응대 보류함. • 폭언, 업무방해 등에 대해 경고문 발송, 고소·고발 등 법적 대응을 추진하여 위법행위를 지속하지   않도록 강력 대응.
[2] metadata
{'_collection_name': 'civil_complaint_manual_medium',
 '_id': '1acb6c40

## 13. 기존 collection 재사용 전용 코드

다음 수업 또는 다음 실행에서 임베딩을 다시 만들고 싶지 않다면 이 코드만 실행한다.  
단, 처음 한 번은 반드시 collection 생성 셀을 실행해야 한다.

In [18]:
reuse_vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    url=QDRANT_URL,
)

reuse_results = reuse_vector_store.similarity_search(question, k=3)

for i, doc in enumerate(reuse_results, start=1):
    print("=" * 80)
    print(f"[{i}] metadata")
    pprint(doc.metadata)
    print("\ncontent preview")
    print(doc.page_content[:500].replace("\n", " "))

[1] metadata
{'_collection_name': 'civil_complaint_manual_medium',
 '_id': 'bfa61ef6-8f1a-433b-9fe5-aa2df965a524',
 'chunk_id': 29,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'medium',
 'collection_name': 'civil_complaint_manual_medium',
 'creationdate': '2025-11-30T19:46:56+09:00',
 'creator': 'Adobe InDesign CC 2014 (Windows)',
 'embedding_model': 'text-embedding-3-small',
 'moddate': '2025-11-30T19:47:04+09:00',
 'page': 16,
 'page_label': '16',
 'producer': 'Adobe PDF Library 11.0',
 'source': '공직자_민원응대_핵심_매뉴얼.pdf',
 'total_pages': 20,
 'trapped': '/False'}

content preview
질의 5 권장시간이 되면 전화를 무조건 끊어야 하는지? • 인·허가 등 정상적인 업무처리 과정에서 민원인과의 상담이 길어지는 경우는 정당한 사유없는 장시간   전화에 해당하지 않으므로 종결대상에 해당하지 않음. 질의 3 상급자 응대 후에도 계속 전화해서 폭언을 하는 경우는 어떻게 대응 해야 하는지? • 담당자에게 휴게시간 부여 등 응대를 중지하도록 조치하고 대행자 등은 다른 민원처리를 위해 해당   민원은 일시적으로 응대 보류함. • 폭언, 업무방해 등에 대해 경고문 발송, 고소·고발 등 법적 대응을 추진하여 위법행위를 지속하지   않도록 강력 대응.
[2] metadata
{'_collection_name': 'civil_complaint_manual_medium',
 '_id': '1acb6c40

## 13. 기존 컬렉션 삭제
### 기존 컬렉션 확인

In [ ]:
# COLLECTION_NAME = "civil_complaint_manual_medium"

# collections = client.get_collections().collections
# collection_names = [collection.name for collection in collections]

# if COLLECTION_NAME in collection_names:
#     client.delete_collection(collection_name=COLLECTION_NAME)
#     print(f"기존 컬렉션 삭제 완료: {COLLECTION_NAME}")
# else:
#     print(f"삭제할 컬렉션이 없습니다: {COLLECTION_NAME}")

# 실습 점검 체크리스트

| 점검 항목 | 확인 방법 |
|---|---|
| Qdrant 도커이미지 확인 | `docker images` |
| Qdrant 컨테이너 실행 여부 | `docker ps` |
| Qdrant Web UI 접속 여부 | `http://localhost:6333/dashboard` |
| collection 생성 여부 | Web UI의 Collections 메뉴 |
| point 수 확인 | `client.get_collection(COLLECTION_NAME)` |
| payload 확인 | Web UI 또는 `client.scroll()` |
| metadata 확인 | `source`, `page`, `chunk_id`, `chunk_strategy` |
| 검색 정상 작동 | `similarity_search_with_score()` |
| 재사용 정상 작동 | `from_existing_collection()` |

# 핵심 정리

- 이번 실습에서는 `medium chunk` 전략만 사용한다.
- 설정값은 `chunk_size=700`, `chunk_overlap=120`이다.
- `QdrantVectorStore.from_documents()`는 문서를 임베딩하고 Qdrant collection에 저장한다.
- `location=":memory:"`를 사용하지 않고 `url="http://localhost:6333"`를 사용하면 Qdrant Server에 저장된다.
- Docker volume을 연결했기 때문에 컨테이너를 재시작해도 임베딩 데이터가 유지된다.
- Qdrant Web UI에서 collection, point, payload, metadata를 직접 확인할 수 있다.
- 이미 저장된 collection은 `QdrantVectorStore.from_existing_collection()`으로 재사용한다.

## 수업용 설명 문장

> Qdrant를 인메모리 방식으로 사용하면 노트북 실행이 끝난 뒤 벡터 데이터가 사라진다.   
> 반면 Docker 기반 Qdrant Server에 저장하면 collection이 유지되고,  
> Web UI에서 각 청크의 원문과 metadata를 직접 확인할 수 있다.   
> 따라서 실제 RAG 실습에서는 medium chunk로 문서를 분할한 뒤 Qdrant Server에 영구 저장하고,   
> 이후에는 기존 collection을 재사용하는 방식이 적합하다.  